# 07 - Final Transformer Training

Train a finalized Transformer on **quantized** token windows from notebook 02.

Assumptions (fixed): context length **256**, stride **128**, cache provides `split_windows['quant']` only.

**Label smoothing** spreads probability mass over non-target classes in cross-entropy, which often reduces overconfident logits and repetitive sampling artifacts.

**Scheduled sampling** (training only) sometimes replaces trailing context tokens with the model’s own greedy predictions before the forward used for loss, so the network sees prefixes closer to what it produces at generation time; val/test loss still uses clean teacher forcing.

Teacher-forced **val loss can improve while free-running generation still sounds poor** (or the reverse), because the training objective is one-step next-token accuracy under ground-truth context, not long-horizon audio quality.


In [ ]:
# Optional for Colab
# !pip install -q torch pandas numpy

In [ ]:
from pathlib import Path
import json
import math
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

PROJECT_OUTPUT_DIR = Path('/content/project_outputs')
CACHE_DIR = PROJECT_OUTPUT_DIR / 'cache'
TABLE_DIR = PROJECT_OUTPUT_DIR / 'tables'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Quant-only cache built with seq_len=256, stride=128 (edit path if needed).
CACHE_PATH = CACHE_DIR / 'sequence_cache.json'

if not CACHE_PATH.exists():
    raise FileNotFoundError(f'Missing cache: {CACHE_PATH}')

with open(CACHE_PATH, 'r') as f:
    cache = json.load(f)

print(f'Loaded cache: {CACHE_PATH}')
V = len(cache['vocab'])
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
# Fixed windowing (must match cache).
SEQ_LEN = 256
STRIDE = 128
        # 'config_id': 't2',
        # 'emb_dim': 192,
        # 'nhead': 6,
        # 'num_layers': 3,
        # 'ff_mult': 4,
        # 'dropout': 0.30,
        # 'lr': 8e-4,
        # 'weight_decay': 5e-4,
        # 'epochs': 10,
        # 'patience': 3,
        # 'batch_size': 64,
        # 'max_train_windows': 4000,
        # 'max_val_windows': 2000,
# Final-run configuration: model + optimization only (from your prior studies).
FINAL_CFG = {
    'run_name': 'final_transformer_quantized_time',

    # Transformer architecture
    'emb_dim': 192,
    'nhead': 6,
    'num_layers': 3,
    'ff_mult': 4,

    # Regularization / optimization
    'dropout': 0.30,
    'lr': 8e-4,
    'weight_decay': 5e-4,
    'batch_size': 64,
    'epochs': 10,
    'patience': 3,

    # Label smoothing (softens one-hot targets; needs torch>=1.10 CrossEntropyLoss support).
    'label_smoothing': 0.05,

    # Scheduled sampling (training only): linear ramp of per-example apply prob; see training cell.
    'scheduled_sampling_max_prob': 0.5,
    'scheduled_sampling_warmup_epochs': 3,
    # k positions at end of context: filled sequentially with greedy argmax(model(x)) each step (no_grad).
    'scheduled_sampling_replace_last': 1,

    # Runtime control (None = full split)
    'max_train_windows': None,
    'max_val_windows': None,
    'max_test_windows': None,
}

print('SEQ_LEN', SEQ_LEN, 'STRIDE', STRIDE)
print(pd.Series(FINAL_CFG))

In [ ]:
def cap_xy(X, y, max_n):
    if max_n is None:
        return X, y
    return X[: int(max_n)], y[: int(max_n)]


def to_loader(X, y, batch_size=64, shuffle=True):
    X_t = torch.tensor(np.array(X), dtype=torch.long)
    y_t = torch.tensor(np.array(y), dtype=torch.long)
    return DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=shuffle)


if 'split_windows' not in cache or 'quant' not in cache['split_windows']:
    raise KeyError("Expected cache['split_windows']['quant'] (quantized windows only).")

swq = cache['split_windows']['quant']
trX, trY = swq['train_X'], swq['train_y']
vaX, vaY = swq['val_X'], swq['val_y']
teX, teY = swq['test_X'], swq['test_y']

trX, trY = cap_xy(trX, trY, FINAL_CFG['max_train_windows'])
vaX, vaY = cap_xy(vaX, vaY, FINAL_CFG['max_val_windows'])
teX, teY = cap_xy(teX, teY, FINAL_CFG['max_test_windows'])

if len(trX) == 0 or len(vaX) == 0 or len(teX) == 0:
    raise RuntimeError('One or more splits are empty after preprocessing/capping.')

if len(trX[0]) != SEQ_LEN:
    raise ValueError(
        f'Expected each training window to have length {SEQ_LEN}, got {len(trX[0])}. '
        'Rebuild sequence cache with seq_len=256.'
    )

if int(cache.get('seq_len', SEQ_LEN)) != SEQ_LEN or int(cache.get('stride', STRIDE)) != STRIDE:
    raise ValueError(
        f"Cache metadata seq_len/stride must be {SEQ_LEN}/{STRIDE}; "
        f"got seq_len={cache.get('seq_len')} stride={cache.get('stride')}."
    )

train_loader = to_loader(trX, trY, batch_size=FINAL_CFG['batch_size'], shuffle=True)
val_loader = to_loader(vaX, vaY, batch_size=FINAL_CFG['batch_size'], shuffle=False)
test_loader = to_loader(teX, teY, batch_size=FINAL_CFG['batch_size'], shuffle=False)

print(f'Windows | train={len(trX)} val={len(vaX)} test={len(teX)}')
print(f'seq_len={SEQ_LEN} stride={STRIDE} (quant only)')

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1), :]


class TransformerNextToken(nn.Module):
    def __init__(self, vocab_size, emb_dim=192, nhead=6, num_layers=3, ff_mult=4, dropout=0.2, max_len=512):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.pos = PositionalEncoding(emb_dim, max_len=max_len)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=nhead,
            dim_feedforward=emb_dim * ff_mult,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.enc = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(emb_dim, vocab_size)

    def forward(self, x):
        h = self.emb(x)
        h = self.pos(h)
        h = self.enc(h)
        h = self.drop(h[:, -1, :])
        return self.fc(h)


def evaluate_loss(model, loader, criterion):
    model.eval()
    losses = []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            losses.append(criterion(model(xb), yb).item())
    return float(np.mean(losses))


def test_metrics(model, loader):
    model.eval()
    all_probs, all_y = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            probs = torch.softmax(model(xb), dim=-1).cpu().numpy()
            all_probs.append(probs)
            all_y.extend(yb.numpy().tolist())

    probs = np.vstack(all_probs)
    preds = probs.argmax(axis=1)
    y = np.array(all_y)
    acc = float(np.mean(y == preds))

    ptrue = np.array([max(probs[i, t], 1e-12) for i, t in enumerate(all_y)])
    ce = float(-np.mean(np.log(ptrue)))
    ppl = float(np.exp(ce))

    k = min(5, probs.shape[1])
    topk = float(np.mean([all_y[i] in np.argpartition(probs[i], -k)[-k:] for i in range(len(all_y))]))
    return {'accuracy': acc, 'cross_entropy': ce, 'perplexity': ppl, 'top5': topk}

In [ ]:
model = TransformerNextToken(
    vocab_size=V,
    emb_dim=FINAL_CFG['emb_dim'],
    nhead=FINAL_CFG['nhead'],
    num_layers=FINAL_CFG['num_layers'],
    ff_mult=FINAL_CFG['ff_mult'],
    dropout=FINAL_CFG['dropout'],
    max_len=max(512, SEQ_LEN),
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=FINAL_CFG['lr'],
    weight_decay=FINAL_CFG['weight_decay'],
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=max(1, FINAL_CFG['epochs'])
)

# Class-weighted CE: increase penalty for larger TIME_SHIFT_n errors.
SHIFT_WEIGHT_ALPHA = 0.08
SHIFT_WEIGHT_MAX = 6.0
MAX_TIME_SHIFT_STEP = 64

id2tok_weight = {int(k): v for k, v in cache['id2tok'].items()}
class_weights = torch.ones(V, dtype=torch.float32)
for tid, tok in id2tok_weight.items():
    if isinstance(tok, str) and tok.startswith('TIME_SHIFT_'):
        try:
            step = int(tok.split('_')[-1])
            # Smooth growth with step size, capped to avoid destabilizing optimization.
            w = 1.0 + SHIFT_WEIGHT_ALPHA * math.log1p(max(1, step))
            class_weights[tid] = min(SHIFT_WEIGHT_MAX, float(w))
        except ValueError:
            pass

# Consecutive TIME_SHIFT cap (default 2): streak counts only tokens generated in this call, not the prompt.
MAX_CONSECUTIVE_TIME_SHIFTS = 2
time_shift_id_mask = torch.zeros(V, dtype=torch.bool, device=device)
for _i in range(V):
    _t = id2tok_weight.get(_i, '')
    if isinstance(_t, str) and _t.startswith('TIME_SHIFT_'):
        time_shift_id_mask[_i] = True

_ls = float(FINAL_CFG.get('label_smoothing', 0.0))


def _make_criterion(label_smoothing: float) -> nn.CrossEntropyLoss:
    kw = dict(weight=class_weights.to(device))
    if label_smoothing > 0.0:
        try:
            return nn.CrossEntropyLoss(**kw, label_smoothing=label_smoothing)
        except TypeError:
            print(
                'WARNING: CrossEntropyLoss(label_smoothing=...) not supported; '
                f'use torch>=1.10.0 (this run: {torch.__version__}). Training without label smoothing.'
            )
            return nn.CrossEntropyLoss(**kw)
    return nn.CrossEntropyLoss(**kw)


criterion = _make_criterion(_ls)


def scheduled_sampling_prob(epoch_1based: int, max_prob: float, warmup_epochs: int) -> float:
    """Linear ramp of per-example apply probability (train only). Reaches max after warmup_epochs."""
    max_prob = float(max_prob)
    warmup_epochs = int(warmup_epochs)
    if max_prob <= 0.0:
        return 0.0
    if warmup_epochs <= 0:
        return max_prob
    if epoch_1based >= warmup_epochs:
        return max_prob
    return max_prob * (float(epoch_1based) / float(warmup_epochs))


def apply_scheduled_sampling_context(xb: torch.Tensor, model: nn.Module, k: int) -> torch.Tensor:
    """Sequentially set xb[:, -1], ..., xb[:, -k] using greedy argmax(model(x)) in no_grad.

    The head predicts the *next* token after the full window only; reusing that distribution
    to fill past slots is a standard cheap approximation (not true per-position marginals).
    """
    k = int(k)
    if k <= 0:
        return xb
    x_sub = xb.clone()
    with torch.no_grad():
        for step in range(k):
            pred = model(x_sub).argmax(dim=-1)
            x_sub[:, -(step + 1)] = pred
    return x_sub


def build_train_batch_inputs(model: nn.Module, xb: torch.Tensor, ss_p_apply: float, k_replace: int) -> torch.Tensor:
    """Per-example Bernoulli(ss_p_apply): if True, use SS-corrupted context for the training forward."""
    if ss_p_apply <= 0.0 or k_replace <= 0:
        return xb
    ss_rows = torch.rand(xb.size(0), device=xb.device) < ss_p_apply
    if not ss_rows.any():
        return xb
    out = xb.clone()
    out[ss_rows] = apply_scheduled_sampling_context(xb[ss_rows], model, k_replace)
    return out


print(f'PyTorch {torch.__version__} | label_smoothing={_ls}')
print(
    f"Weighted CE enabled | alpha={SHIFT_WEIGHT_ALPHA} cap={SHIFT_WEIGHT_MAX} "
    f"| max_weight={class_weights.max().item():.3f}"
)
print(
    f"Scheduled sampling (train) | max_prob={FINAL_CFG['scheduled_sampling_max_prob']} "
    f"warmup_epochs={FINAL_CFG['scheduled_sampling_warmup_epochs']} "
    f"replace_last_k={FINAL_CFG['scheduled_sampling_replace_last']}"
)
# Sanity: after two generated TIME_SHIFT_* in a row, the third generated token cannot be TIME_SHIFT_* (streak resets on any non-shift).
print(
    f"Decode caps | per-step TIME_SHIFT_n <= {MAX_TIME_SHIFT_STEP} | "
    f"max_consecutive_shifts={MAX_CONSECUTIVE_TIME_SHIFTS} (generated suffix only)"
)


def sample_transformer_capped_timeshift(
    model,
    prompt_ids,
    steps,
    temperature=1.0,
    seq_len=256,
    max_time_shift_step=MAX_TIME_SHIFT_STEP,
    max_consecutive_time_shifts=MAX_CONSECUTIVE_TIME_SHIFTS,
):
    """
    Two decoding rules (streak applies only to newly generated tokens, not the prompt):

    1) Per-step cap: if sampled TIME_SHIFT_n has n > max_time_shift_step, mask that id once and resample.

    2) Consecutive cap: after max_consecutive_time_shifts generated TIME_SHIFT_* in a row, mask all
       TIME_SHIFT_* for that step; if softmax degenerates, argmax on working logits (deterministic).
    """
    out = list(prompt_ids)
    streak = 0

    for _ in range(int(steps)):
        x = torch.tensor([out[-seq_len:]], dtype=torch.long, device=device)
        with torch.no_grad():
            logits = model(x)[0] / max(float(temperature), 1e-6)

        # Working logits: consecutive-shift rule may zero out every TIME_SHIFT id for this step only.
        log_work = logits.clone()
        if streak >= int(max_consecutive_time_shifts):
            log_work = log_work.masked_fill(time_shift_id_mask, float('-inf'))

        probs = torch.softmax(log_work, dim=-1)
        if torch.isnan(probs).any() or (probs.sum().item() <= 0) or not torch.isfinite(probs).all():
            idx = int(torch.argmax(log_work).item())
        else:
            idx = int(torch.multinomial(probs, num_samples=1).item())

        tok = id2tok_weight.get(idx, '')
        # Per-step TIME_SHIFT_n cap (independent of streak): forbid oversize n for one resample.
        if isinstance(tok, str) and tok.startswith('TIME_SHIFT_'):
            try:
                step = int(tok.split('_')[-1])
                if step > int(max_time_shift_step):
                    log2 = log_work.clone()
                    log2[idx] = float('-inf')
                    p2 = torch.softmax(log2, dim=-1)
                    if torch.isfinite(p2).all() and p2.sum().item() > 0 and not torch.isnan(p2).any():
                        idx = int(torch.multinomial(p2, num_samples=1).item())
                    else:
                        idx = int(torch.argmax(log2).item())
            except ValueError:
                pass

        if bool(time_shift_id_mask[idx].item()):
            streak += 1
        else:
            streak = 0
        out.append(idx)

    return out[len(prompt_ids):]


def evaluate_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb).argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    return float(correct / max(total, 1))


import time

best_state = None
best_val_loss = np.inf
best_epoch = 0
no_improve = 0
history = []
t_run0 = time.perf_counter()

for epoch in range(1, FINAL_CFG['epochs'] + 1):
    t_ep0 = time.perf_counter()
    model.train()
    train_losses = []
    train_correct = 0
    train_total = 0

    p_ss = scheduled_sampling_prob(
        epoch,
        FINAL_CFG['scheduled_sampling_max_prob'],
        FINAL_CFG['scheduled_sampling_warmup_epochs'],
    )
    k_ss = int(FINAL_CFG['scheduled_sampling_replace_last'])

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        xb_in = build_train_batch_inputs(model, xb, p_ss, k_ss)
        logits = model(xb_in)
        loss = criterion(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_losses.append(loss.item())
        preds = logits.argmax(dim=1)
        train_correct += (preds == yb).sum().item()
        train_total += yb.size(0)

    scheduler.step()

    train_loss = float(np.mean(train_losses))
    train_acc = float(train_correct / max(train_total, 1))
    val_loss = evaluate_loss(model, val_loader, criterion)
    val_acc = evaluate_accuracy(model, val_loader)
    lr_now = float(optimizer.param_groups[0]['lr'])
    ep_sec = time.perf_counter() - t_ep0

    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'train_acc': train_acc,
        'val_acc': val_acc,
        'lr': lr_now,
        'scheduled_sampling_p': p_ss,
    })

    improved = val_loss < best_val_loss - 1e-5
    print(
        f"[epoch {epoch:02d}/{FINAL_CFG['epochs']}] "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
        f"lr={lr_now:.2e} | "
        f"ss_p={p_ss:.3f} | "
        f"best_val={best_val_loss:.4f}@ep{best_epoch} | "
        f"no_improve={no_improve}/{FINAL_CFG['patience']} | "
        f"epoch_time={ep_sec:.1f}s | "
        f"elapsed={time.perf_counter() - t_run0:.1f}s",
        flush=True,
    )

    if improved:
        best_val_loss = val_loss
        best_epoch = epoch
        no_improve = 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"  -> new best val_loss={best_val_loss:.4f} (saved state)", flush=True)
    else:
        no_improve += 1
        if no_improve >= FINAL_CFG['patience']:
            print(
                f"Early stopping at epoch {epoch} (patience={FINAL_CFG['patience']}, "
                f"best val_loss={best_val_loss:.4f} at epoch {best_epoch}).",
                flush=True,
            )
            break

if best_state is not None:
    model.load_state_dict(best_state)

history_df = pd.DataFrame(history)
metrics = test_metrics(model, test_loader)
metrics.update({
    'run_name': FINAL_CFG['run_name'],
    'setting': 'quantized_time',
    'seq_len': SEQ_LEN,
    'stride': STRIDE,
    'best_epoch': best_epoch,
    'best_val_loss': best_val_loss,
    'n_train_windows': len(trX),
    'n_val_windows': len(vaX),
    'n_test_windows': len(teX),
    'emb_dim': FINAL_CFG['emb_dim'],
    'nhead': FINAL_CFG['nhead'],
    'num_layers': FINAL_CFG['num_layers'],
    'ff_mult': FINAL_CFG['ff_mult'],
    'dropout': FINAL_CFG['dropout'],
    'lr': FINAL_CFG['lr'],
    'weight_decay': FINAL_CFG['weight_decay'],
    'batch_size': FINAL_CFG['batch_size'],
    'epochs_requested': FINAL_CFG['epochs'],
    'patience': FINAL_CFG['patience'],
    'loss_name': 'weighted_cross_entropy',
    'shift_weight_alpha': SHIFT_WEIGHT_ALPHA,
    'shift_weight_cap': SHIFT_WEIGHT_MAX,
    'decode_max_time_shift_step': MAX_TIME_SHIFT_STEP,
    'decode_max_consecutive_time_shifts': MAX_CONSECUTIVE_TIME_SHIFTS,
    'pytorch_version': torch.__version__,
    'label_smoothing': float(FINAL_CFG.get('label_smoothing', 0.0)),
    'scheduled_sampling_max_prob': float(FINAL_CFG['scheduled_sampling_max_prob']),
    'scheduled_sampling_warmup_epochs': int(FINAL_CFG['scheduled_sampling_warmup_epochs']),
    'scheduled_sampling_replace_last': int(FINAL_CFG['scheduled_sampling_replace_last']),
})

metrics_df = pd.DataFrame([metrics])
display(history_df.tail(10))
display(metrics_df)

In [ ]:
ts = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
run_slug = f"{ts}__{FINAL_CFG['run_name']}"

history_path = TABLE_DIR / f"07_training_history_{run_slug}.csv"
metrics_path = TABLE_DIR / f"07_training_metrics_{run_slug}.csv"
ckpt_path = CACHE_DIR / f"07_transformer_{run_slug}.pt"

history_df.to_csv(history_path, index=False)
metrics_df.to_csv(metrics_path, index=False)
torch.save(model.state_dict(), ckpt_path)

print('Saved outputs:')
print(' -', history_path)
print(' -', metrics_path)
print(' -', ckpt_path)

## Notes

- Cache must be built with **quant** windows only in `split_windows['quant']`, **seq_len=256**, **stride=128**.
- For a full-data run, set `max_train_windows`, `max_val_windows`, and `max_test_windows` to `None`.
- Point your Gradio demo at the saved `.pt` checkpoint and use **the same** `seq_len=256` there.


In [ ]:
# Drop-in demo-candidate exporter: per-example test metrics + MIDI outputs
from pathlib import Path
from datetime import datetime
import math
import re

import numpy as np
import pandas as pd
import pretty_midi
import torch
import torch.nn.functional as F

TOP_K = 10
DEMO_DIR = Path("data/project_outputs/demo_candidates")
MIN_NOTE_DUR = 0.01
DEMO_DIR.mkdir(parents=True, exist_ok=True)

# Use notebook settings when available.
time_bin = float(cache.get("time_bin_sec", cache.get("time_bin", 0.01)))

# Resolve run slug safely (do not overwrite blindly).
_export_slug = globals().get("run_slug")
if not _export_slug:
    _export_slug = datetime.utcnow().strftime("%Y%m%d_%H%M%S") + "__candidate_export"

# Robust id->token map (cache JSON may store keys as strings).
id2tok_raw = cache.get("id2tok", {})
if isinstance(id2tok_raw, dict):
    id2tok_map = {}
    for k, v in id2tok_raw.items():
        try:
            id2tok_map[int(k)] = v
        except Exception:
            continue
else:
    id2tok_map = {}

if len(id2tok_map) == 0:
    raise RuntimeError("Could not build id2tok map from cache; required for MIDI export.")

# 1) Per-example metrics aligned with teX/teY order.
model.eval()
rows = []
batch_size_eval = int(FINAL_CFG.get("batch_size", 64))
N = len(teX)

with torch.no_grad():
    for start in range(0, N, batch_size_eval):
        end = min(start + batch_size_eval, N)
        xb_np = np.array(teX[start:end], dtype=np.int64)
        yb_np = np.array(teY[start:end], dtype=np.int64)

        xb = torch.tensor(xb_np, dtype=torch.long, device=device)
        yb = torch.tensor(yb_np, dtype=torch.long, device=device)

        logits = model(xb)
        probs = torch.softmax(logits, dim=-1)

        losses = F.cross_entropy(logits, yb, reduction="none")
        confs, preds = probs.max(dim=-1)

        losses_np = losses.detach().cpu().numpy()
        confs_np = confs.detach().cpu().numpy()
        preds_np = preds.detach().cpu().numpy()

        for j in range(end - start):
            idx = start + j
            y_true = int(yb_np[j])
            y_pred = int(preds_np[j])
            rows.append(
                {
                    "index": idx,
                    "target_id": y_true,
                    "pred_id": y_pred,
                    "is_correct": int(y_true == y_pred),
                    "loss": float(losses_np[j]),
                    "confidence": float(confs_np[j]),
                }
            )

per_example_df = pd.DataFrame(rows)
if per_example_df.empty:
    raise RuntimeError("No per-example rows computed from test set.")

# 2) Rank: correct first, then low loss, then high confidence.
ranked_df = per_example_df.sort_values(
    by=["is_correct", "loss", "confidence", "index"],
    ascending=[False, True, False, True],
).reset_index(drop=True)

correct_ranked = ranked_df[ranked_df["is_correct"] == 1].copy()
if len(correct_ranked) > 0:
    selected_df = correct_ranked.head(TOP_K).copy()
else:
    # Fallback: no correct predictions available.
    selected_df = ranked_df.sort_values(by=["loss", "confidence", "index"], ascending=[True, False, True]).head(TOP_K).copy()

# 3) Convert selected contexts to MIDI.
def _safe_token_lookup(token_id: int) -> str:
    tok = id2tok_map.get(int(token_id))
    if isinstance(tok, str):
        return tok
    return "UNK"


_time_shift_pat = re.compile(r"^TIME_SHIFT_(\d+)$")
_note_on_pat = re.compile(r"^NOTE_ON_(\d+)$")
_note_off_pat = re.compile(r"^NOTE_OFF_(\d+)$")


def context_end_seconds(tokens, time_bin_sec=0.01):
    t = 0.0
    for tok in tokens:
        if not isinstance(tok, str):
            continue
        m = _time_shift_pat.match(tok)
        if m is not None:
            t += max(1, int(m.group(1))) * time_bin_sec
    return float(t)


def tokens_to_pretty_midi(tokens, time_bin_sec=0.01, min_note_dur=0.01):
    pm = pretty_midi.PrettyMIDI()
    inst = pretty_midi.Instrument(program=0)

    current_t = 0.0
    # pitch -> list[(start, velocity)] to handle overlapping same-pitch NOTE_ON.
    active = {}

    for tok in tokens:
        if not isinstance(tok, str):
            continue

        m = _time_shift_pat.match(tok)
        if m is not None:
            step = max(1, int(m.group(1)))
            current_t += step * time_bin_sec
            continue

        m = _note_on_pat.match(tok)
        if m is not None:
            pitch = int(m.group(1))
            if 0 <= pitch <= 127:
                active.setdefault(pitch, []).append((current_t, 100))
            continue

        m = _note_off_pat.match(tok)
        if m is not None:
            pitch = int(m.group(1))
            if pitch in active and len(active[pitch]) > 0:
                start_t, velocity = active[pitch].pop(0)
                end_t = max(current_t, start_t + min_note_dur)
                inst.notes.append(
                    pretty_midi.Note(
                        velocity=int(velocity),
                        pitch=int(pitch),
                        start=float(start_t),
                        end=float(end_t),
                    )
                )
            # Guard: NOTE_OFF without NOTE_ON is ignored.
            continue

    # Close any dangling active notes.
    for pitch, starts in active.items():
        for start_t, velocity in starts:
            end_t = start_t + min_note_dur
            inst.notes.append(
                pretty_midi.Note(
                    velocity=int(velocity),
                    pitch=int(pitch),
                    start=float(start_t),
                    end=float(end_t),
                )
            )

    pm.instruments.append(inst)
    return pm


manifest_rows = []
for rank, row in enumerate(selected_df.itertuples(index=False), start=1):
    idx = int(row.index)
    if idx < 0 or idx >= len(teX):
        continue

    ctx_ids = teX[idx]
    target_id = int(row.target_id)
    full_ids = list(ctx_ids) + [target_id]

    # Export sequence = context window + ground-truth next token.
    full_tokens = [_safe_token_lookup(int(tid)) for tid in full_ids]
    ctx_end_sec = context_end_seconds([_safe_token_lookup(int(tid)) for tid in ctx_ids], time_bin_sec=time_bin)
    ctx_end_tag = f"{ctx_end_sec:.2f}".replace(".", "p")
    pm = tokens_to_pretty_midi(full_tokens, time_bin_sec=time_bin, min_note_dur=MIN_NOTE_DUR)

    midi_name = (
        f"{_export_slug}__candidate_rank{rank:02d}_"
        f"idx{idx}_ctxend{ctx_end_tag}s_corr{int(row.is_correct)}_loss{float(row.loss):.4f}.mid"
    )
    midi_path = DEMO_DIR / midi_name
    pm.write(str(midi_path))

    manifest_rows.append(
        {
            "rank": rank,
            "test_index": idx,
            "is_correct": int(row.is_correct),
            "loss": float(row.loss),
            "confidence": float(row.confidence),
            "pred_id": int(row.pred_id),
            "target_id": target_id,
            "context_end_sec": float(ctx_end_sec),
            "midi_path": str(midi_path),
            "n_tokens": int(len(full_ids)),
            "n_context_tokens": int(len(ctx_ids)),
            "n_appended_gt_tokens": 1,
        }
    )

manifest_df = pd.DataFrame(manifest_rows)
manifest_path = DEMO_DIR / f"{_export_slug}__demo_candidates_manifest.csv"
manifest_df.to_csv(manifest_path, index=False)

print(f"Output directory: {DEMO_DIR.resolve()}")
print(f"Wrote MIDI files: {len(manifest_df)}")
print(f"Manifest: {manifest_path}")
if len(manifest_df) > 0:
    display(manifest_df.head(3))
else:
    print("No demo candidates exported.")